# CS4241 - Introduction to Artificial Intelligence
## Part G: Innovation Component (6 Marks)

**Name:** Maureen Amago  
**Index Number:** 10022200180

----- 
### 🚀 Novel Feature: Domain-Specific Scoring Function (Section-Aware Boosting)

**Innovation Overview:** 
Standard RAG models use simple math (Cosine Similarity) to find text. However, a Budget Document has a strict structure. This innovation adds a **"Budget Intelligence Layer"** that identifies the user's intent and applies a **Domain Weight** to chunks from relevant sections.

---
## Cell 1: Setup & Data Loading

In [ ]:
# Name: Maureen Amago | Index: 10022200180
import re, math, numpy as np
from pypdf import PdfReader
from collections import Counter

# Load PDF and identify sections
reader = PdfReader('2025-Budget-Statement-and-Economic-Policy_v4.pdf')
chunks = []
chunk_metadata = []

raw_text = ''
for i in range(min(50, len(reader.pages))):
    text = reader.pages[i].extract_text() or ""
    
    # NOVELTY: Assign metadata based on section headers found in text
    section = "General"
    if "Revenue" in text: section = "Revenue"
    elif "Expenditure" in text: section = "Expenditure"
    elif "Macroeconomic" in text or "GDP" in text: section = "Macroeconomics"
    
    # Clean and Chunk
    clean_p = re.sub(r'\s+', ' ', text).strip()
    for j in range(0, len(clean_p), 450):
        chunk = clean_p[j:j+500]
        chunks.append(chunk)
        chunk_metadata.append({"section": section})

print(f"✅ Loaded {len(chunks)} chunks with Domain Metadata.")

---
## Cell 2: Domain-Specific Search Engine
This engine doesn't just look for words; it looks for **intent**.

In [ ]:
# Name: Maureen Amago | Index: 10022200180

class InnovativeSearch:
    def __init__(self, docs, metadata):
        self.docs = docs
        self.metadata = metadata
        all_tok = re.findall(r'\b\w{2,}\b', " ".join(docs).lower())
        self.vocab = {w: i for i, (w, _) in enumerate(Counter(all_tok).most_common(2000))}
        self.idf = {w: math.log(len(docs)/(1+sum(1 for d in docs if w in d.lower()))) for w in self.vocab}
        self.vecs = np.array([self._embed(d) for d in docs])

    def _embed(self, text):
        v = np.zeros(len(self.vocab))
        toks = re.findall(r'\b\w{2,}\b', text.lower())
        for w, c in Counter(toks).items():
            if w in self.vocab: v[self.vocab[w]] = (c / len(toks)) * self.idf[w]
        return v

    def search(self, query, k=3):
        # 1. Detect Intent
        intent = "General"
        if any(w in query.lower() for w in ["tax", "revenue", "levy", "income"]): intent = "Revenue"
        elif any(w in query.lower() for w in ["spend", "allocation", "cost", "expenditure"]): intent = "Expenditure"
        elif any(w in query.lower() for w in ["growth", "gdp", "inflation"]): intent = "Macroeconomics"
        
        # 2. Base Cosine Similarity
        qv = self._embed(query)
        scores = []
        for i, cv in enumerate(self.vecs):
            d = np.linalg.norm(qv) * np.linalg.norm(cv)
            base_sim = np.dot(qv, cv) / d if d > 0 else 0
            
            # 3. INNOVATION: Apply Domain-Specific Boost
            boost = 1.0
            if self.metadata[i]["section"] == intent and intent != "General":
                boost = 1.5  # 50% boost for relevant sections!
            
            final_score = base_sim * boost
            scores.append((final_score, i, base_sim, boost))
            
        top = sorted(scores, reverse=True)[:k]
        return top

innovative_vs = InnovativeSearch(chunks, chunk_metadata)
print("✅ Domain-Specific Scoring Engine Ready.")

---
## Cell 3: Evidence of Innovation
We compare the search results for a query about taxes with and without the domain boost.

In [ ]:
# Name: Maureen Amago | Index: 10022200180

test_query = "How is the government increasing tax revenue?"
results = innovative_vs.search(test_query)

print(f"QUERY: {test_query}\n")
print(f"{ 'Rank' : <5} | { 'Final Score' : <12} | { 'Base Sim' : <10} | { 'Boost' : <6} | { 'Section' : <15}")
print("-"*60)

for rank, (final, idx, base, boost) in enumerate(results):
    section = chunk_metadata[idx]['section']
    print(f"{ rank+1 : <5} | { final : <12.4f} | { base : <10.4f} | { boost : <6.1f} | { section : <15}")

print("\nOBSERVATION: Chunks from the 'Revenue' section received a 1.5x boost, ensuring the most relevant domain context is selected.")